# 01 Hypothesis validation

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
from itertools import combinations
sys.path.append(str(Path("..")))
from src.ingest.open_parquet import abrir_parquet_data_pandas 




In [5]:
df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv') 

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\avazq\AppData\Local\Temp\ipykernel_22992\3771880710.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv')


In [6]:
df_candidate_pairs

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES
...,...,...,...
185,SHOP,TEAM,SERVICES-PREPACKAGED SOFTWARE
186,SHOP,TTWO,SERVICES-PREPACKAGED SOFTWARE
187,SNPS,TEAM,SERVICES-PREPACKAGED SOFTWARE
188,SNPS,TTWO,SERVICES-PREPACKAGED SOFTWARE


## Research Decision: Correlation Definition

For pair ranking we define:

- Observation frequency: 1-minute
- Variable: log returns
- Correlation: Pearson correlation
- Window: 60 trading days (23,460 observations)

Lead-lag relationships will be evaluated separately in Hypothesis 1 following Lo & MacKinlay (1990).

The correct workflow would be:

 1. Read each cleared stock by symbol

 2. For each symbol: 
     - calculate log_return_1m = log(close_t / close_t-1)

 3. For each pair in candidate_pairs:
     - merge stock_a and stock_b by date

4. Calculate the 60-day rolling correlation
     - window = 60 × 391 = 23,460 observations

5. For each day:
     - obtain the most recent available rolling correlation

6. Rank all pairs by correlation

7. Select the Top 50

In [ ]:
DATA_DIR = Path("../data/rth_1min_by_symbol")
OUT_DIR = Path("../data/rth_1min_log_returns_by_symbol")
OUT_DIR.mkdir(exist_ok=True)

def add_log_returns(symbol_file):
    df = pd.read_parquet(symbol_file)
    df = df.sort_values("date").copy()
    df["log_return_1m"] = np.log(df["close"] / df["close"].shift(1))

    return df


,date,symbol,close,log_return_1m
0,2021-06-01 09:30:00,AAPL,125.269997,NaN
1,2021-06-01 09:31:00,AAPL,125.189102,-0.000646
2,2021-06-01 09:32:00,AAPL,124.988998,-0.001600
3,2021-06-01 09:33:00,AAPL,124.849998,-0.001113
4,2021-06-01 09:34:00,AAPL,124.860001,0.000080
5,2021-06-01 09:35:00,AAPL,124.830002,-0.000240
6,2021-06-01 09:36:00,AAPL,124.754997,-0.000601
7,2021-06-01 09:37:00,AAPL,124.848801,0.000752
8,2021-06-01 09:38:00,AAPL,124.855003,0.000050
9,2021-06-01 09:39:00,AAPL,124.779999,-0.000601


In [10]:
# save parquets with log returns
for file in DATA_DIR.glob("*_rth_1min.parquet"):
    symbol = file.name.replace("_rth_1min.parquet", "")

    df_ret = add_log_returns(file)

    df_ret.to_parquet(
        OUT_DIR / f"{symbol}_rth_1min_returns.parquet",
        index=False
    )

In [11]:
sample = pd.read_parquet(OUT_DIR / "NVDA_rth_1min_returns.parquet")

sample[["date", "symbol", "close", "log_return_1m"]].head()

,date,symbol,close,log_return_1m
0,2021-06-01 09:30:00,NVDA,16.251499,NaN
1,2021-06-01 09:31:00,NVDA,16.256500,0.000308
2,2021-06-01 09:32:00,NVDA,16.221300,-0.002168
3,2021-06-01 09:33:00,NVDA,16.180799,-0.002500
4,2021-06-01 09:34:00,NVDA,16.153601,-0.001682


In [12]:
sample["log_return_1m"].isna().sum()

np.int64(1)

Pair Selection Methodology

For each trading day t:

1. Use the previous 60 trading days of 1-minute log returns.
2. Compute Pearson correlation for every candidate pair.
3. Rank all candidate pairs by correlation.
4. Select the Top 50 pairs.
5. Use this Top 50 universe for signal generation during day t.